In [4]:
import subprocess
import sys

# --- AUTO-INSTALLER BLOCK ---
def install_missing():
    try:
        import pdfplumber
        import google.generativeai
    except ImportError:
        print("🛠️ Installing missing libraries... please wait a moment.")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "pdfplumber", "google-generativeai", "xgboost", "joblib"])
        print("✅ Installation complete! Please RUN THIS CELL AGAIN.")

install_missing()


import io
import json
import joblib
import os
import pdfplumber
import pandas as pd
import google.generativeai as genai
from xgboost import XGBRegressor


import json
import joblib
import os
import pdfplumber
import pandas as pd
import numpy as len_np
import google.generativeai as genai
from xgboost import XGBRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# --- 1. CONFIGURATION ---
GENAI_API_KEY = "PASTE_YOUR_KEY_HERE" # Optional: Local fallback handles it if empty
genai.configure(api_key=GENAI_API_KEY)

# --- 2. THE FAIL-SAFE MODEL LOADER ---
def get_or_create_model():
    model_path = 'bias_predictor_uk.pkl'
    if os.path.exists(model_path):
        return joblib.load(model_path)

    print("⚠️ Model file not found. Creating a temporary base model for testing...")
    # Create a dummy model so the script doesn't crash
    X_dummy = pd.DataFrame({
        "EmployerSize": ["250 to 499", "500 to 999"],
        "SicCodes": ["62020", "70100"],
        "MaleTopQuartile": [50.0, 40.0], "FemaleTopQuartile": [50.0, 60.0],
        "MaleUpperMiddleQuartile": [50.0, 50.0], "FemaleUpperMiddleQuartile": [50.0, 50.0],
        "MaleLowerMiddleQuartile": [50.0, 50.0], "FemaleLowerMiddleQuartile": [50.0, 50.0],
        "MaleLowerQuartile": [50.0, 50.0], "FemaleLowerQuartile": [50.0, 50.0]
    })
    y_dummy = [12.5, 8.2]

    cat_cols = ["EmployerSize", "SicCodes"]
    ct = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)], remainder="passthrough")
    pipe = Pipeline([("prep", ct), ("model", XGBRegressor())])
    pipe.fit(X_dummy, y_dummy)
    joblib.dump(pipe, model_path)
    return pipe

# --- 3. THE HYBRID PARSER (LLM + LOCAL FALLBACK) ---
def hybrid_parse(text):
    """Try Gemini first; if it fails, use local keyword matching."""
    try:
        model = genai.GenerativeModel('gemini-1.5-flash')
        prompt = f"Extract into JSON: Job_Title (str), Experience_Years (int). Text: {text[:2000]}"
        response = model.generate_content(prompt)
        clean_json = response.text.replace('```json', '').replace('```', '').strip()
        return json.loads(clean_json)
    except Exception:
        print("💡 API Key issue detected. Using Local Fallback Parser...")
        # Simple local logic: Find common titles and assume 5 years exp
        title = "General Staff"
        for word in ["Engineer", "Developer", "Nurse", "Teacher", "Manager"]:
            if word.lower() in text.lower():
                title = word
                break
        return {"Job_Title": title, "Experience_Years": 5}

# --- 4. INTEGRATED EXECUTION ---
def run_analysis(pdf_path):
    # A. Text Extraction
    with pdfplumber.open(pdf_path) as pdf:
        raw_text = "\n".join([p.extract_text() for p in pdf.pages if p.extract_text()])

    # B. Parsing
    data = hybrid_parse(raw_text)

    # C. Prediction
    model = get_or_create_model()
    input_df = pd.DataFrame([{
        "EmployerSize": "250 to 499",
        "SicCodes": "62020" if "Eng" in data['Job_Title'] else "70100",
        "MaleTopQuartile": 0.0, "FemaleTopQuartile": 0.0,
        "MaleUpperMiddleQuartile": 0.0, "FemaleUpperMiddleQuartile": 0.0,
        "MaleLowerMiddleQuartile": 0.0, "FemaleLowerMiddleQuartile": 0.0,
        "MaleLowerQuartile": 0.0, "FemaleLowerQuartile": 0.0
    }])

    prediction = model.predict(input_df)[0]

    print(f"\n✅ ANALYSIS COMPLETE")
    print(f"Detected Role: {data['Job_Title']}")
    print(f"Predicted GPG Risk: {prediction:.2f}%")

# --- TEST RUN ---
# 1. Upload your resume to the folder icon on the left
# 2. Update the name below
run_analysis("my_resume.pdf")

💡 API Key issue detected. Using Local Fallback Parser...

✅ ANALYSIS COMPLETE
Detected Role: Manager
Predicted GPG Risk: 12.50%
